# 03 — Préparation modèle (features sortie + télé)

**Objectif :** construire les matrices de features pour **score_sortie** et **score_tele**, vérifier les colonnes clés, exporter les parquets curated.

> Prérequis : notebook `02_nettoyage_donnees` (ou sources CSV à la racine).  
> Suite : `04_entrainement_modele.ipynb`


## 1. Imports & chemins


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "patients.csv").is_file() and (p / "Sujet.md").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

from src.data.paths import ProjectPaths
from notebooks._utils.schema import FEATURES_CLES, SCHEMA_ATTENDU

paths = ProjectPaths(root=ROOT)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
print("Racine :", ROOT)
print("Curated :", paths.curated)
print("Models  :", paths.models)


## 2. Rechargement / nettoyage minimal (reproductible)


In [ ]:
import shutil

from src.data.cohort import filter_domicile, patient_level_split
from src.data.identity import persist_identity_split, split_identity_and_pseudonymise
from src.data.load import copy_sources_to_raw, load_csv
from src.data.paths import CSV_FILES
from src.data.quality import (
    filter_objets_connectes_bons,
    harmonize_biologie,
    recalculate_duree_sejour,
)

copy_sources_to_raw(paths)
patients = load_csv(paths, "patients.csv")
vault, patients_pseudo = split_identity_and_pseudonymise(patients)
persist_identity_split(paths, vault, patients_pseudo)

for name in CSV_FILES:
    if name == "patients.csv":
        continue
    shutil.copy2(paths.raw / name, paths.pseudonymise / name)

sejours_el = patient_level_split(filter_domicile(recalculate_duree_sejour(load_csv(paths, "sejours.csv"))))
historique = load_csv(paths, "historique.csv")
biologies = harmonize_biologie(load_csv(paths, "biologies.csv"))
signes = load_csv(paths, "signes_vitaux.csv")
diagnostics = load_csv(paths, "diagnostics.csv")
actes = load_csv(paths, "actes.csv")
medications = load_csv(paths, "medications.csv")
territoire = load_csv(paths, "territoire_insee.csv")
objets = filter_objets_connectes_bons(load_csv(paths, "objets_connectes.csv"))

print("sejours_el:", len(sejours_el), "| objets bons:", len(objets))


## 3. Features score_sortie → DataFrame + vérif colonnes


In [ ]:
from src.data.features_sortie import build_features_score_sortie

features_sortie = build_features_score_sortie(
    sejours_el,
    patients_pseudo,
    historique,
    biologies,
    signes,
    diagnostics,
    actes,
    medications,
    territoire,
)

manquantes = [c for c in FEATURES_CLES if c not in features_sortie.columns]
if manquantes:
    raise AssertionError(f"Colonnes clés manquantes (sortie) : {manquantes}")

print("features_sortie :", features_sortie.shape)
print("Colonnes :", list(features_sortie.columns))
display(features_sortie.head(3))


## 4. Features score_tele (horizon 7 j) → DataFrame + vérif


In [ ]:
from src.data.features_tele import build_features_score_tele

HORIZON_TELE = 7
features_tele = build_features_score_tele(
    features_sortie, sejours_el, objets, horizon_jours=HORIZON_TELE
)

manquantes = [c for c in FEATURES_CLES if c not in features_tele.columns]
if manquantes:
    raise AssertionError(f"Colonnes clés manquantes (télé) : {manquantes}")

extra = [c for c in features_tele.columns if c not in features_sortie.columns]
print("features_tele :", features_tele.shape)
print("Colonnes ajoutées télé :", extra)
display(features_tele.head(3))


## 5. Export curated + splits X/y


In [ ]:
from src.model.prepare import by_split, split_xy

paths.ensure_data_dirs()
p_el = paths.curated / "sejours_eligibles.parquet"
p_so = paths.curated / "features_score_sortie.parquet"
p_te = paths.curated / "features_score_tele.parquet"

sejours_el.to_parquet(p_el, index=False)
features_sortie.to_parquet(p_so, index=False)
features_tele.to_parquet(p_te, index=False)

print("Écrit :", p_el)
print("Écrit :", p_so)
print("Écrit :", p_te)

rows = []
for nom, feat in [("score_sortie", features_sortie), ("score_tele", features_tele)]:
    X, y, split = split_xy(feat)
    for part in ("train", "val", "test"):
        Xp, yp = by_split(X, y, split, part)
        rows.append({
            "score": nom,
            "split": part,
            "n": len(Xp),
            "prevalence": round(float(yp.mean()), 4),
            "n_features": Xp.shape[1],
        })

display(pd.DataFrame(rows))
print("✓ Préparation terminée — enchaînez avec 04_entrainement_modele.ipynb")
